In [57]:
!wget -O 11-patient-characteristics-cluster.csv https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/11-patient-characteristics-cluster.csv

--2026-04-20 20:46:56--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/11-patient-characteristics-cluster.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 319760 (312K) [application/octet-stream]
Saving to: ‘11-patient-characteristics-cluster.csv’

11-patient-characte 100%[===================>] 312.27K  --.-KB/s    in 0.003s  

2026-04-20 20:46:57 (105 MB/s) - ‘11-patient-characteristics-cluster.csv’ saved [319760/319760]



In [58]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import Row
import math
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline
from pyspark.sql.functions import monotonically_increasing_id
from pyspark.ml.clustering import KMeans

In [59]:
conf = SparkConf().setMaster("local").setAppName("Assignment11")
# Stop existing SparkContext if it exists, to prevent 'ValueError: Cannot run multiple SparkContexts at once'
if 'SpContext' in globals() and SpContext:
    SpContext.stop()

# Configure Spark and create SparkContext and SparkSession
SpContext = SparkContext(conf=conf)
SpSession = SparkSession.builder.appName("YourAppName").config("spark.sql.warehouse.dir", "/content").getOrCreate()

characData = SpContext.textFile("11-patient-characteristics-cluster.csv")
characData.cache()

# Remove the first line (contains headers)
firstLine = characData.first()
dataLines = characData.filter(lambda x: x != firstLine)
dataLines.count()

4003

In [60]:
# 1. Prepare Data
# Convert RDD of text lines to a DataFrame with an 'id' and 'text' column
text_df = dataLines.map(lambda line: (line,)).toDF(["text"])

In [62]:
# 2. Text Normalization Pipeline (TF-IDF)
from pyspark.sql.functions import monotonically_increasing_id

# Add 'id' to the dataframe
text_df_with_id = text_df.withColumn("id", monotonically_increasing_id())

# Configure the TF-IDF pipeline
tokenizer = Tokenizer(inputCol="text", outputCol="words")
stopwordsRemover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=10000)
idf = IDF(inputCol="rawFeatures", outputCol="features")

# Create the pipeline
tfidf_pipeline = Pipeline(stages=[tokenizer, stopwordsRemover, hashingTF, idf])

# Fit and transform
tfidf_model = tfidf_pipeline.fit(text_df_with_id)
tfidf_data = tfidf_model.transform(text_df_with_id)

print("TF-IDF features generated:")
tfidf_data.select("id", "text", "features").show(5, truncate=False)

TF-IDF features generated:
+---+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id |text                                                                                                                                                                                            |features                                                  

In [64]:
# 3. K-Means Clustering
# Parameters: K=10, seed=1
kmeans = KMeans(k=10, seed=1, featuresCol="features", predictionCol="cluster")
kmeans_model = kmeans.fit(tfidf_data)

# 4. Generate Outcome
results = kmeans_model.transform(tfidf_data)

print("Cluster Assignment Results:")
results.select("text", "cluster").show(10, truncate=80)

# Calculate Cost (Within Set Sum of Squared Errors)
cost = kmeans_model.summary.trainingCost
print("\n" + f"Within Set Sum of Squared Errors: {cost}")

Cluster Assignment Results:
+--------------------------------------------------------------------------------+-------+
|                                                                            text|cluster|
+--------------------------------------------------------------------------------+-------+
|Current substance or alcohol use disorder as determined by the SCID or by pos...|      0|
|                             Subjects who consume >14 alcoholic drinks per week.|      0|
|                               Excessive consumption of xanthine-based beverages|      0|
|History of drug abuse or use of illegal drugs: use of soft drugs (marijuana  ...|      0|
|                                                                    Non-smokers.|      0|
|Smokers may participate  but they are limited to 10 cigarettes per day while ...|      0|
|Regular use of alcohol within six months prior to the screening visit (more t...|      0|
|Use of soft drugs ( such as marijuana) within 3 months prior 